In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
# Task 1: Write your code here:
data = os.path.join(path, 'Q3_data.csv')
data = pd.read_csv(data)


In [ ]:
# Task 2: Write your code here:
data.head()

In [ ]:
# Task 3: Write your code here:
data.info()

In [ ]:
# Task 4: Write your code here:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
# Task 1: Write your code here:
numerical_cols = data.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

for col in numerical_cols:
  data[col] = data[col].fillna(data[col].mode()[0])

data



In [ ]:
# Task 2: Write your code here:
data.duplicated().sum()
data.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = data.select_dtypes(include=["object"]).columns


label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  data[col] = le.fit_transform(data[col])
  label_encoders[col] = le

data

In [ ]:

from sklearn.preprocessing import StandardScaler
# Task 4: Write your code here:


scaler = StandardScaler()
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])
data.head()


In [ ]:
# Task 5: Write your code here:

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(data['Target'].dropna(), bins=30, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split, StratifiedKFold

X = data.drop("Target" , axis=1)
y = data["Target"]

n_splits = 3 # K=3 Folds

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
n_splits = 5 # K=3 Folds

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

sr_results = {'loss': [], 'acc': [], 'f1': []}

model = CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


    # Fit the model on train data
  model.fit(X_train, y_train)

    # Use the model to predict the test data
  y_pred = model.predict(X_test)

    # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  sr_results['acc'].append(accuracy)
  sr_results['f1'].append(f1)

print(f"  Accuracy:  {np.mean(sr_results['acc']):.4f}")
print(f"  F1-Score:  {np.mean(sr_results['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:
importances = {}


importances['CatBoost'] = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_feature = model.feature_importances_
print(np.argsort(golden_feature))

In [ ]:
# Task Bonus: Write your code here: